# 08 - Chế độ trích xuất (PhoBERT) trên 30 câu dev (Kaggle / Colab)

**Mục đích.** Kiểm chế độ trích xuất của Tuần 3 T4 trên môi trường thật (Java, VnCoreNLP, GPU, index BM25 + BGE-M3): truy hồi hybrid của B, rồi PhoBERT đọc các đoạn truy hồi và chọn một khoảng văn bản làm câu trả lời (`src/extractive.py`, `src/pipeline.py::answer_question_extractive`). Notebook cũng gọi một lần hàm hiển thị của `app/app.py`; ô cuối (tùy chọn) mở demo Gradio.

**Phạm vi.**
- Chỉ dùng `gold_dev.jsonl`; dataset không chứa `gold_test.jsonl` và notebook kiểm tra điều này.
- Cấu hình truy hồi (top-5, `rrf_k = 60`, không xếp hạng lại theo tiêu đề) là mặc định chưa chốt, giống notebook 07; đổi hằng ở ô cấu hình nếu B chốt khác.
- Ngưỡng từ chối cố định `NULL_THRESHOLD = 0.0`. Không tinh chỉnh ngưỡng trên 30 câu này. Ngưỡng -1,489326 (LR 5e-5, tinh chỉnh trên ViQuAD Dev, `docs/results/qa_week3_A_training_summary.md`) chỉ được tính lại từ `score_diff` đã lưu để xem ngưỡng có chuyển được sang miền Quy chế hay không.
- Số EM/F1 chỉ có nghĩa khi dùng checkpoint thật của A. Không có checkpoint (`REAL_CKPT = None`) thì notebook dùng đầu QA khởi tạo ngẫu nhiên: chỉ kiểm không lỗi, định dạng và độ trễ; mọi số về chất lượng đều vô nghĩa.

**Điều kiện trước khi chạy**
1. Một Dataset duy nhất `vi-qa-rag-extractive` gồm mã nguồn (từ commit `79aa301`: `src/`, `app/`, `scripts/build_index.py`, `scripts/run_eval.py`, PDF Quy chế, `gold_dev.jsonl`, `gold_split.json`) và thư mục `checkpoint/phobert_qa_lr5e-5_seed12/best_model` (PhoBERT LR 5e-5 do A huấn luyện). Dataset không chứa `gold_test.jsonl` và repo không chứa trọng số.
2. Không có checkpoint trong Dataset thì notebook dùng đầu QA ngẫu nhiên (chỉ kiểm không lỗi). Notebook tự tìm `best_model/qa_head.pt` trong `/kaggle/input`; điền tay vào `REAL_CKPT` nếu có nhiều bản.
3. Session: GPU T4, Internet bật (tải BGE-M3, VnCoreNLP, `vinai/phobert-base` nếu dùng đầu ngẫu nhiên, cài Java).
4. Chạy *Run All*; tải `extractive_dev_*.{json,jsonl,csv}` trong `/kaggle/working/` về `docs/results/extractive_dev/`.

## 1. Môi trường

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_MODE"] = "disabled"
if os.path.isdir("/kaggle/temp"):
    os.environ["HF_HOME"] = "/kaggle/temp/hf"

# transformers ghim 4.57.6 như requirements.txt và notebook 06 (đã chạy tokenizer PhoBERT trên Kaggle với bản này).
!pip install -q transformers==4.57.6 sentence-transformers rank-bm25==0.2.2 pdfplumber py_vncorenlp==0.1.4
import shutil
if shutil.which("java") is None:
    !apt-get -qq update && apt-get -qq install -y default-jre-headless
!java -version 2>&1 | head -1

In [ ]:
import csv, glob, hashlib, importlib.metadata as md, importlib.util, json, platform, statistics, sys, time
from pathlib import Path

import torch, transformers

assert torch.cuda.is_available(), "Không có GPU. Chọn Accelerator là GPU T4 trong Session options."
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu_name": props.name, "gpu_total_gb": round(props.total_memory / 1e9, 2),
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "sentence_transformers": md.version("sentence-transformers"),
    "py_vncorenlp": md.version("py_vncorenlp"),
}
print(json.dumps(ENV, indent=2))

## 2. Cấu hình

In [ ]:
# Thư mục best_model của A (chứa config.json, qa_head.pt, qa_config.json, tokenizer). None = đầu QA ngẫu nhiên.
REAL_CKPT = None   # ví dụ: "/kaggle/input/phobert-lr5e-5/best_model"

TOP_K, RRF_K = 5, 60
NULL_THRESHOLD = 0.0                 # ngưỡng từ chối của lần chạy chính
VIQUAD_TUNED_THRESHOLD = -1.489326   # LR 5e-5, tinh chỉnh trên ViQuAD Dev; chỉ để tính lại, không dùng làm mặc định
LAUNCH_DEMO = False                  # True: ô cuối mở demo Gradio (share=True) và chặn đến khi dừng ô

## 3. Nạp mã nguồn và dữ liệu từ Dataset

`/kaggle/input` chỉ đọc, trong khi `build_index.py` ghi index vào `data/index/`, nên cả thư mục được sao sang ổ ghi được. Mã băm SHA-256 của từng file `.py` và của `gold_dev.jsonl` được ghi vào file kết quả để đối chiếu với commit.

In [ ]:
matches = glob.glob("/kaggle/input/**/src/extractive.py", recursive=True) + glob.glob("./**/src/extractive.py", recursive=True)
assert matches, "Không tìm thấy src/extractive.py. Hãy gắn Dataset mã nguồn bằng 'Add Input' (Kaggle) hoặc tải thư mục dataset lên cạnh notebook (Colab)."
DATASET_ROOT = Path(matches[0]).resolve().parents[1]
REPO = (Path("/kaggle/temp/repo") if Path("/kaggle/temp").is_dir() else Path("repo")).resolve()
OUT = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")).resolve()
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(DATASET_ROOT, REPO, ignore=shutil.ignore_patterns("*.safetensors", "*.pt"))   # bỏ trọng số, không cần trong REPO

for required in ("src/extractive.py", "src/pipeline.py", "src/retrieval.py", "src/preprocessing_qa.py", "src/qa_nbest.py",
                 "src/qa_metrics.py", "app/app.py", "scripts/build_index.py", "scripts/run_eval.py",
                 "data/domain/dhqghn_quy_che_dao_tao_dai_hoc_2014.pdf", "data/gold/gold_dev.jsonl", "data/splits/gold_split.json"):
    assert (REPO / required).exists(), f"Dataset thiếu file: {required}"
assert not (REPO / "data" / "gold" / "gold_test.jsonl").exists(), "Dataset không được chứa gold_test.jsonl"

CODE_SNAPSHOT = {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()[:12]
                 for p in sorted(REPO.glob("src/*.py")) + sorted(REPO.glob("scripts/*.py")) + sorted(REPO.glob("app/*.py"))}
GOLD_PATH = REPO / "data" / "gold" / "gold_dev.jsonl"
GOLD_SHA = hashlib.sha256(GOLD_PATH.read_bytes()).hexdigest()[:12]
print("dataset:", DATASET_ROOT, "| gold_dev sha256:", GOLD_SHA)

## 4. VnCoreNLP

In [ ]:
import py_vncorenlp

VN_DIR = REPO / "data" / "raw" / "vncorenlp"
VN_DIR.mkdir(parents=True, exist_ok=True)
py_vncorenlp.download_model(save_dir=str(VN_DIR))
os.chdir(REPO)   # download_model có thể đổi thư mục làm việc
assert (VN_DIR / "VnCoreNLP-1.2.jar").is_file(), os.listdir(VN_DIR)
print(sorted(os.listdir(VN_DIR)))

## 5. Dựng index Quy chế

Chạy đúng `scripts/build_index.py --corpus domain` của B. Số chunk in ra cần là 216 (kho hiện hành của nhóm).

In [ ]:
!cd {REPO} && python scripts/build_index.py --corpus domain 2>&1 | grep -v -i "it/s\|Batches" | tail -n 25

## 6. Checkpoint PhoBERT

`REAL_CKPT = None` thì notebook tìm `best_model/qa_head.pt` trong `/kaggle/input`; điền tay nếu có nhiều bản. Không có checkpoint nào thì tạo checkpoint đúng định dạng `train_qa.py` lưu (`backbone.save_pretrained`, `tokenizer.save_pretrained`, `qa_head.pt`, `qa_config.json`) với đầu QA ngẫu nhiên.

In [ ]:
if not REAL_CKPT:   # tự tìm best_model trong Dataset đã gắn (Kaggle giải nén zip khi tạo Dataset)
    found = sorted(glob.glob("/kaggle/input/**/best_model/qa_head.pt", recursive=True))
    print("best_model tìm thấy:", [str(Path(f).parent) for f in found])
    if found:
        REAL_CKPT = str(Path(found[0]).parent)
if REAL_CKPT:
    CKPT = Path(REAL_CKPT)
    print("DÙNG CHECKPOINT THẬT:", CKPT)
else:
    from transformers import AutoModel, AutoTokenizer
    CKPT = Path("/kaggle/working/random_ckpt/best_model") if Path("/kaggle/working").is_dir() else Path("random_ckpt/best_model")
    backbone = AutoModel.from_pretrained("vinai/phobert-base")
    backbone.save_pretrained(CKPT)
    AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False).save_pretrained(CKPT)
    torch.manual_seed(0)
    torch.save(torch.nn.Linear(backbone.config.hidden_size, 2).state_dict(), CKPT / "qa_head.pt")
    (CKPT / "qa_config.json").write_text(json.dumps({"hidden_size": backbone.config.hidden_size, "max_answer_length": 30, "null_threshold": 0.0}))
    del backbone
    print("DÙNG ĐẦU QA NGẪU NHIÊN: mọi số về chất lượng bên dưới vô nghĩa")
print(sorted(os.listdir(CKPT)))
IS_REAL_CKPT = bool(REAL_CKPT)

## 7. Nạp retriever, văn bản các đoạn, 30 câu dev và mô hình

Mã đoạn có dạng `dieu21_khoan3`; một Khoản dài có thể tách thành nhiều chunk (`_diemX`, `_wN`), và mọi chunk đó đều tính là Khoản đúng. Ô này kiểm tra giống notebook 07: đủ 30 câu (27 trong phạm vi, 3 ngoài phạm vi), `id` trùng đúng `dev_ids` trong `data/splits/gold_split.json`, mỗi câu trong phạm vi có chunk của Khoản đúng trong index.

In [ ]:
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from scripts.run_eval import load_bm25, load_dense
from src.extractive import load_extractive_reader, make_extract_fn
from src.pipeline import answer_question_extractive
from src.qa_metrics import exact_match_score, f1_score

bm25 = load_bm25("domain")
dense = load_dense("domain")
CHUNK_TEXTS = {item["id"]: item["text"] for item in json.load(open(REPO / "data/index/domain/chunks_meta.json", encoding="utf-8"))}
print("chunks:", len(CHUNK_TEXTS))

ITEMS = sorted((json.loads(line) for line in open(GOLD_PATH, encoding="utf-8-sig") if line.strip()), key=lambda x: x["id"])
n_in = sum(bool(i["in_scope"]) for i in ITEMS)
assert len(ITEMS) == 30 and n_in == 27, "gold_dev.jsonl không khớp kỳ vọng 30 câu (27 trong phạm vi)"
assert all(i.get("split") == "dev" for i in ITEMS), "Có câu không thuộc split dev"
assert {i["id"] for i in ITEMS} == set(json.load(open(REPO / "data/splits/gold_split.json", encoding="utf-8"))["dev_ids"])


def gold_prefix(item):
    return f"dieu{item['article']}_khoan{item['clause']}"


def is_gold_id(doc_id, prefix):
    return doc_id == prefix or doc_id.startswith(prefix + "_")


for item in ITEMS:
    if item["in_scope"]:
        assert item["clause"] is not None, f"{item['id']}: thiếu Khoản"
        assert any(is_gold_id(i, gold_prefix(item)) for i in CHUNK_TEXTS), f"{item['id']}: không có chunk của Khoản đúng"

# JVM của VnCoreNLP chỉ khởi động được một lần mỗi kernel: chạy lại ô này sau khi đã nạp thì phải Restart kernel.
assert "READER" not in globals(), "READER đã được nạp; Restart kernel rồi Run All để nạp lại VnCoreNLP."
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
READER = load_extractive_reader(CKPT, VN_DIR, device="cuda")
os.chdir(REPO)   # phòng khi py_vncorenlp đổi thư mục làm việc
extract_fn = make_extract_fn(READER, null_threshold=NULL_THRESHOLD)
LOAD = {"load_seconds": round(time.perf_counter() - t0, 1), "vram_allocated_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
        "max_answer_length": READER["max_answer_length"]}
print(LOAD)

## 8. Chạy chế độ trích xuất trên 30 câu dev

Mỗi câu chạy một lần, thứ tự theo `id`; một câu khởi động chạy trước để JVM, CUDA và BGE-M3 không làm lệch số đo của câu đầu. `source_is_gold`: đoạn chứa câu trả lời thuộc Khoản đúng. `first_gold_rank`: thứ hạng đầu tiên của chunk thuộc Khoản đúng trong top-k.

In [ ]:
_ = answer_question_extractive(extract_fn, bm25, dense, CHUNK_TEXTS, "Khởi động", top_k=TOP_K, rrf_k=RRF_K)


def run_item(item):
    torch.cuda.reset_peak_memory_stats()
    res = answer_question_extractive(extract_fn, bm25, dense, CHUNK_TEXTS, item["question"], top_k=TOP_K, rrf_k=RRF_K)
    torch.cuda.synchronize()
    prefix = gold_prefix(item) if item["in_scope"] else None
    ranks = [r for r, i in enumerate(res["sources"], start=1) if prefix and is_gold_id(i, prefix)]
    return {
        "id": item["id"], "in_scope": bool(item["in_scope"]), "question": item["question"],
        "gold_answers": item["answers"], "answer": res["answer"], "refusal": res["refusal"],
        "best_text": res["nbest"][0]["text"] if res["nbest"] else "", "score_diff": res["score_diff"],
        "source_id": res["source_id"], "source_rank": res["source_rank"],
        "source_is_gold": bool(prefix and res["source_id"] and is_gold_id(res["source_id"], prefix)),
        "first_gold_rank": min(ranks) if ranks else None, "sources": res["sources"], "windows": res["windows"],
        "retrieval_seconds": res["retrieval_seconds"], "extract_seconds": res["seconds"],
        "vram_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


ROWS = [run_item(item) for item in ITEMS]
for r in ROWS:
    print(f"[{r['id']}] refusal={r['refusal']} | in_scope={r['in_scope']} | gold rank: {r['first_gold_rank']} | source: {r['source_id']}")
    print("  Q:", r["question"])
    print("  A:", r["answer"].replace("\n", " "), "| gold:", r["gold_answers"][:1])

## 9. Tổng hợp

EM/F1 theo cách của `scripts/run_b1_domain_dev.py`: câu trong phạm vi so với các đáp án gold; câu ngoài phạm vi có tham chiếu là chuỗi rỗng và dự đoán rỗng khi từ chối. Hai ngưỡng được tính lại từ `score_diff` đã lưu (không chạy lại mô hình). Đây là 30 câu dev đã dùng để thiết kế nên không phải kết quả đánh giá cuối.

In [ ]:
def score_at(threshold):
    def pred(r):
        return r["best_text"] if r["score_diff"] is not None and r["score_diff"] <= threshold else ""

    def refs(r):
        return r["gold_answers"] if r["in_scope"] else [""]

    def group(rows):
        if not rows:
            return {"n": 0, "em": None, "f1": None}
        return {"n": len(rows),
                "em": round(100 * sum(float(exact_match_score(pred(r), refs(r))) for r in rows) / len(rows), 2),
                "f1": round(100 * sum(float(f1_score(pred(r), refs(r))) for r in rows) / len(rows), 2)}

    return {"threshold": threshold, "all": group(ROWS), "in_scope": group([r for r in ROWS if r["in_scope"]]),
            "out_of_scope": group([r for r in ROWS if not r["in_scope"]]),
            "answered": sum(pred(r) != "" for r in ROWS)}


in_rows = [r for r in ROWS if r["in_scope"]]
SUMMARY = {
    "checkpoint": "real" if IS_REAL_CKPT else "RANDOM_HEAD (số chất lượng vô nghĩa)",
    "n_items": len(ROWS), "n_in_scope": len(in_rows), "n_out_of_scope": len(ROWS) - len(in_rows),
    "in_scope_refused": sum(r["refusal"] for r in in_rows),
    "out_of_scope_refused": sum(r["refusal"] for r in ROWS if not r["in_scope"]),
    "gold_in_top_k": {f"top{k}": sum(r["first_gold_rank"] is not None and r["first_gold_rank"] <= k for r in in_rows) for k in (1, 3, 5)},
    "source_is_gold": sum(r["source_is_gold"] for r in in_rows),
    "scores": {"fixed_0": score_at(NULL_THRESHOLD), "viquad_tuned": score_at(VIQUAD_TUNED_THRESHOLD)},
    "median_retrieval_seconds": round(statistics.median(r["retrieval_seconds"] for r in ROWS), 3),
    "median_extract_seconds": round(statistics.median(r["extract_seconds"] for r in ROWS), 3),
    "max_extract_seconds": max(r["extract_seconds"] for r in ROWS),
    "median_windows": statistics.median(r["windows"] for r in ROWS),
    "max_vram_peak_gb": max(r["vram_peak_gb"] for r in ROWS),
}
print(json.dumps(SUMMARY, indent=2, ensure_ascii=False))

## 10. Hàm hiển thị của demo (và demo Gradio tùy chọn)

In [ ]:
spec = importlib.util.spec_from_file_location("demo_app", REPO / "app" / "app.py")
demo_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo_app)
answer_fns = {demo_app.MODE_EXTRACT: lambda q: answer_question_extractive(extract_fn, bm25, dense, CHUNK_TEXTS, q, top_k=TOP_K, rrf_k=RRF_K)}
shown_answer, shown_sources = demo_app.format_result(answer_fns[demo_app.MODE_EXTRACT](ITEMS[0]["question"]), CHUNK_TEXTS)
print(shown_answer, "\n---\n", shown_sources)

# Gradio chạy hàm trả lời trong luồng phụ; kiểm VnCoreNLP (pyjnius) và PhoBERT gọi được từ luồng khác luồng chính.
import threading
box = {}
worker = threading.Thread(target=lambda: box.update(res=answer_fns[demo_app.MODE_EXTRACT](ITEMS[0]["question"])))
worker.start()
worker.join()
assert box.get("res"), "Luồng phụ không trả kết quả"
print("gọi từ luồng phụ: OK |", box["res"]["answer"][:80])

In [ ]:
if LAUNCH_DEMO:
    !pip install -q gradio
    demo_app.build_demo(answer_fns, CHUNK_TEXTS).launch(share=True)   # chặn đến khi dừng ô; mở liên kết public in ra

## 11. Lưu kết quả

Ba file trong `/kaggle/working/`: `extractive_dev_results.json` (môi trường, cấu hình, mã băm, tổng hợp, từng câu), `extractive_dev_answers.jsonl` (`id`, `answer`), `extractive_dev_items.csv`.

In [ ]:
RESULTS = {
    "env": ENV, "code_snapshot": CODE_SNAPSHOT, "gold_dev_sha256_12": GOLD_SHA,
    "config": {"retrieval": "hybrid (BM25 + BGE-M3, RRF)", "top_k": TOP_K, "rrf_k": RRF_K, "title_rerank": False,
               "null_threshold": NULL_THRESHOLD, "checkpoint": str(CKPT), "real_checkpoint": IS_REAL_CKPT,
               "n_chunks": len(CHUNK_TEXTS)},
    "load": LOAD, "summary": SUMMARY, "items": ROWS,
}
(OUT / "extractive_dev_results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")
with open(OUT / "extractive_dev_answers.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for r in ROWS:
        f.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")
fields = ["id", "in_scope", "question", "answer", "refusal", "score_diff", "source_id", "source_rank", "source_is_gold",
          "first_gold_rank", "windows", "retrieval_seconds", "extract_seconds", "vram_peak_gb"]
with open(OUT / "extractive_dev_items.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ROWS)
print("saved:", sorted(p.name for p in OUT.glob("extractive_dev_*")))

## 12. Cách đọc kết quả

| Quan sát | Ý nghĩa | Việc tiếp theo |
|---|---|---|
| Ô 7 hoặc 8 báo lỗi | Lỗi môi trường hoặc mã (Java, VnCoreNLP, căn chỉnh token trên văn bản thật, khớp checkpoint) | Gửi nguyên văn lỗi cho người viết mã |
| `out_of_scope_refused` < 3 | Câu ngoài phạm vi vẫn được trích một khoảng văn bản: PhoBERT luôn có khoảng ứng viên, chỉ ngưỡng quyết định từ chối | Xem `score_diff` của 3 câu này so với câu trong phạm vi để biết có ngưỡng nào phân tách được; không chọn ngưỡng trên 30 câu rồi báo như kết quả |
| `in_scope_refused` cao | Ngưỡng 0 từ chối quá nhiều, hoặc điểm logit giữa các đoạn chưa hiệu chỉnh | So hai mục `scores.fixed_0` và `scores.viquad_tuned` |
| `source_is_gold` thấp hơn `gold_in_top_k.top5` | PhoBERT chọn khoảng ở đoạn sai dù Khoản đúng có trong top-k | Ghi vào phân tích lỗi (Tuần 4 T4) |
| `max_vram_peak_gb` | Bộ nhớ khi chỉ có PhoBERT và BGE-M3; Qwen 4-bit cần thêm khoảng 5-6 GB khi chạy demo hai chế độ | Nếu sát 15 GB, chuyển BGE-M3 sang CPU khi chạy demo |